In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r"C:\Users\soulx\Downloads\web_traffic.csv")

print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:")
print(df.isnull().sum())

   SessionID        Date      PagePath         Source DeviceCategory  \
0  SESS-5001  2026-06-10      /landing  Google Search         Mobile   
1  SESS-5002  2026-06-10      /courses         Direct        Desktop   
2  SESS-5003  2026-06-10  /blog/python       LinkedIn         Mobile   
3  SESS-5004  2026-06-11     /register     Google Ads        Desktop   
4  SESS-5005  2026-06-11      /pricing         Direct        Desktop   

   SessionDurationSec  PageViews Converted  BounceRate  
0                 145          3       Yes         0.0  
1                 420          8       Yes         0.0  
2                  45          1        No         1.0  
3                 210          4       Yes         0.0  
4                  60          2        No         0.0  

Shape: (10, 9)

Columns: ['SessionID', 'Date', 'PagePath', 'Source', 'DeviceCategory', 'SessionDurationSec', 'PageViews', 'Converted', 'BounceRate']

Missing Values:
SessionID             0
Date                  0
PagePath  

In [12]:
print(df["Converted"].unique())
print(df["Converted"].dtype)

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
str


In [14]:
df["Converted"] = df["Converted"].map({"Yes": 1, "No": 0})

conversion_rate = df["Converted"].mean() * 100

print("===== WEB TRAFFIC SUMMARY =====")
print("Total Sessions:", len(df))
print("Bounce Rate:", round(df["BounceRate"].mean() * 100, 2), "%")
print("Average Session Duration:", round(df["SessionDurationSec"].mean(), 2), "seconds")
print("Average Pageviews:", round(df["PageViews"].mean(), 2))
print("Overall Conversion Rate:", round(conversion_rate, 2), "%")

===== WEB TRAFFIC SUMMARY =====
Total Sessions: 10
Bounce Rate: 30.0 %
Average Session Duration: 170.5 seconds
Average Pageviews: 3.4
Overall Conversion Rate: 50.0 %


In [15]:
source_analysis = df.groupby("Source").agg(
    Sessions=("SessionID", "count"),
    Conversions=("Converted", "sum"),
    AvgSessionDuration=("SessionDurationSec", "mean"),
    AvgPageViews=("PageViews", "mean"),
    AvgBounceRate=("BounceRate", "mean")
)

source_analysis["ConversionRate"] = (
    source_analysis["Conversions"] / source_analysis["Sessions"] * 100
)

print(source_analysis.round(2))

               Sessions  Conversions  AvgSessionDuration  AvgPageViews  \
Source                                                                   
Direct                3            1               220.0          4.33   
Facebook              1            0                15.0          1.00   
Google Ads            2            2               250.0          4.50   
Google Search         2            2               227.5          4.50   
LinkedIn              1            0                45.0          1.00   
Twitter               1            0                30.0          1.00   

               AvgBounceRate  ConversionRate  
Source                                        
Direct                   0.0           33.33  
Facebook                 1.0            0.00  
Google Ads               0.0          100.00  
Google Search            0.0          100.00  
LinkedIn                 1.0            0.00  
Twitter                  1.0            0.00  


In [16]:
source_group = df.copy()

source_group["TrafficChannel"] = source_group["Source"].map({
    "Google Search": "Organic",
    "Google Ads": "Ads",
    "Facebook": "Social",
    "LinkedIn": "Social",
    "Twitter": "Social",
    "Direct": "Direct"
})

channel_analysis = source_group.groupby("TrafficChannel").agg(
    Sessions=("SessionID", "count"),
    Conversions=("Converted", "sum"),
    AvgSessionDuration=("SessionDurationSec", "mean"),
    AvgPageViews=("PageViews", "mean"),
    AvgBounceRate=("BounceRate", "mean")
)

channel_analysis["ConversionRate"] = (
    channel_analysis["Conversions"] / channel_analysis["Sessions"] * 100
)

print(channel_analysis.round(2))

                Sessions  Conversions  AvgSessionDuration  AvgPageViews  \
TrafficChannel                                                            
Ads                    2            2               250.0          4.50   
Direct                 3            1               220.0          4.33   
Organic                2            2               227.5          4.50   
Social                 3            0                30.0          1.00   

                AvgBounceRate  ConversionRate  
TrafficChannel                                 
Ads                       0.0          100.00  
Direct                    0.0           33.33  
Organic                   0.0          100.00  
Social                    1.0            0.00  


In [17]:
page_analysis = df.groupby("PagePath").agg(
    Sessions=("SessionID", "count"),
    Conversions=("Converted", "sum"),
    AvgPageViews=("PageViews", "mean"),
    AvgSessionDuration=("SessionDurationSec", "mean"),
    AvgBounceRate=("BounceRate", "mean")
)

page_analysis["ConversionRate"] = (
    page_analysis["Conversions"] / page_analysis["Sessions"] * 100
)

page_analysis["DropOffRate"] = 100 - page_analysis["ConversionRate"]

print(page_analysis.sort_values("DropOffRate", ascending=False).round(2))

                       Sessions  Conversions  AvgPageViews  \
PagePath                                                     
/blog/ai-trends               1            0           1.0   
/blog/python                  1            0           1.0   
/contact                      1            0           3.0   
/pricing                      1            0           2.0   
/landing                      3            2           3.0   
/courses                      1            1           8.0   
/courses/data-science         1            1           6.0   
/register                     1            1           4.0   

                       AvgSessionDuration  AvgBounceRate  ConversionRate  \
PagePath                                                                   
/blog/ai-trends                      30.0           1.00            0.00   
/blog/python                         45.0           1.00            0.00   
/contact                            180.0           0.00            0.00   

In [18]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages("web_traffic_dashboard.pdf") as pdf:

    fig = plt.figure(figsize=(11, 8.5))
    fig.suptitle("Web Traffic Analytics Dashboard", fontsize=20, fontweight="bold")

    metrics = [
        ("Total Sessions", len(df)),
        ("Bounce Rate", f"{df['BounceRate'].mean()*100:.1f}%"),
        ("Avg Session Duration", f"{df['SessionDurationSec'].mean():.1f} sec"),
        ("Avg Pageviews", f"{df['PageViews'].mean():.1f}"),
        ("Conversion Rate", f"{df['Converted'].mean()*100:.1f}%")
    ]

    for i, (label, value) in enumerate(metrics):
        ax = fig.add_axes([0.05 + i*0.19, 0.65, 0.16, 0.15])
        ax.text(0.5, 0.6, str(value), ha="center", va="center", fontsize=20, fontweight="bold")
        ax.text(0.5, 0.2, label, ha="center", va="center", fontsize=10)
        ax.axis("off")

    ax = fig.add_axes([0.08, 0.12, 0.38, 0.40])
    channel_analysis["ConversionRate"].plot(kind="bar", ax=ax)
    ax.set_title("Conversion Rate by Channel")
    ax.set_ylabel("Conversion Rate (%)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

    ax = fig.add_axes([0.55, 0.12, 0.38, 0.40])
    channel_analysis["AvgBounceRate"].mul(100).plot(kind="bar", ax=ax)
    ax.set_title("Bounce Rate by Channel")
    ax.set_ylabel("Bounce Rate (%)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 8.5))

    page_plot = page_analysis.sort_values("DropOffRate", ascending=False)

    page_plot["DropOffRate"].plot(kind="bar", ax=ax)

    ax.set_title("Page-Level Drop-Off Rate")
    ax.set_ylabel("Drop-Off Rate (%)")
    ax.set_xlabel("Page")
    ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

print("Dashboard created: web_traffic_dashboard.pdf")

Dashboard created: web_traffic_dashboard.pdf
